# System Design II: Data and ML Systems

Notebook for **Chapter 9** of the *career* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/career/ch09.html

Tip: change a number, predict the output, then run the cell again.

## 9.1 The ML Design Framework and the Data Platform

In [ ]:
events = 1_000_000_000      # per day
size = 1000                 # bytes per event, raw JSON
print("events/s avg :", round(events / 86400))
print("events/s peak:", round(events / 86400 * 3))
raw_tb = events * size / 1e12
print("raw TB/day   :", raw_tb, " per year:", raw_tb * 365)
comp = 5
print("columnar TB/year at 5x:", raw_tb * 365 / comp)
mbps_peak = events / 86400 * 3 * size / 1e6
print("peak MB/s:", round(mbps_peak, 1), "-> partitions at 10 MB/s each:", -(-mbps_peak // 10))
batch_hour = 3600
print("worst-case staleness hourly batch + 10 min job:", (batch_hour + 600) / 60, "min")

events/s avg : 11574
events/s peak: 34722
raw TB/day   : 1.0  per year: 365.0
columnar TB/year at 5x: 73.0
peak MB/s: 34.7 -> partitions at 10 MB/s each: 4.0
worst-case staleness hourly batch + 10 min job: 70.0 min


## 9.2 Feature Pipelines and Training-Serving Skew

In [ ]:
import sqlite3
c = sqlite3.connect(":memory:")
c.executescript("""
create table feat(user_id int, ts text, avg_spend real);
insert into feat values (1,'2026-01-01',100),(1,'2026-01-10',500),(2,'2026-01-05',80);
create table label(user_id int, ts text, churned int);
insert into label values (1,'2026-01-08',0),(1,'2026-01-12',1),(2,'2026-01-04',0);
""")
print("WRONG (latest feature, leaks future):")
for r in c.execute("""select l.user_id,l.ts,l.churned,
  (select avg_spend from feat f where f.user_id=l.user_id order by ts desc limit 1) from label l order by 1,2"""): print(r)
print("RIGHT (latest feature at or before label time):")
for r in c.execute("""select l.user_id,l.ts,l.churned,
  (select avg_spend from feat f where f.user_id=l.user_id and f.ts<=l.ts order by ts desc limit 1) from label l order by 1,2"""): print(r)

WRONG (latest feature, leaks future):
(1, '2026-01-08', 0, 500.0)
(1, '2026-01-12', 1, 500.0)
(2, '2026-01-04', 0, 80.0)
RIGHT (latest feature at or before label time):
(1, '2026-01-08', 0, 100.0)
(1, '2026-01-12', 1, 500.0)
(2, '2026-01-04', 0, None)


## 9.3 Worked Design: Recommender System

In [ ]:
stages = [("retrieval ANN", 1_000_000, 500, 15), ("light ranker", 500, 100, 10),
          ("heavy ranker", 100, 20, 40), ("rules + diversity", 20, 10, 5)]
total = 0
for name, n_in, n_out, ms in stages:
    total += ms
    print(f"{name:18s} {n_in:>9,} -> {n_out:>4}  {ms:3d} ms  cumulative {total} ms")
relevant = {3, 7, 9}
ranked = [7, 1, 3, 4, 5, 9, 2]
for k in (2, 4, 6):
    hits = len(relevant & set(ranked[:k]))
    print(f"precision@{k} = {hits}/{k} = {hits/k:.2f}   recall@{k} = {hits}/{len(relevant)} = {hits/len(relevant):.2f}")
items = 1_000_000; d = 64
print("item embeddings MB (float32, d=64):", items * d * 4 / 1e6)

retrieval ANN      1,000,000 ->  500   15 ms  cumulative 15 ms
light ranker             500 ->  100   10 ms  cumulative 25 ms
heavy ranker             100 ->   20   40 ms  cumulative 65 ms
rules + diversity         20 ->   10    5 ms  cumulative 70 ms
precision@2 = 1/2 = 0.50   recall@2 = 1/3 = 0.33
precision@4 = 2/4 = 0.50   recall@4 = 2/3 = 0.67
precision@6 = 3/6 = 0.50   recall@6 = 3/3 = 1.00
item embeddings MB (float32, d=64): 256.0


## 9.4 Worked Design: Fraud Detection

In [ ]:
tps = 5000; fraud_rate = 0.001
print("fraud tx/s:", tps * fraud_rate, " per day:", tps * fraud_rate * 86400)
for recall, fpr in ((0.8, 0.005), (0.8, 0.001), (0.9, 0.02)):
    tp = recall * fraud_rate
    fp = fpr * (1 - fraud_rate)
    prec = tp / (tp + fp)
    flagged_per_s = tps * (tp + fp)
    print(f"recall {recall} fpr {fpr}: precision {prec:.3f}, flagged/s {flagged_per_s:.1f}, false alarms/s {tps*fp:.1f}")

fraud tx/s: 5.0  per day: 432000.0
recall 0.8 fpr 0.005: precision 0.138, flagged/s 29.0, false alarms/s 25.0
recall 0.8 fpr 0.001: precision 0.445, flagged/s 9.0, false alarms/s 5.0
recall 0.9 fpr 0.02: precision 0.043, flagged/s 104.4, false alarms/s 99.9


In [ ]:
loss_per_fraud = 8000; review_cost = 40
n = 1_000_000
frauds = n * fraud_rate
for name, recall, fpr in (("A", 0.8, 0.005), ("B", 0.9, 0.02)):
    caught = frauds * recall
    fa = (n - frauds) * fpr
    net = caught * loss_per_fraud - (caught + fa) * review_cost
    print(name, "caught", caught, "false alarms", round(fa), "net saving", round(net))

A caught 800.0 false alarms 4995 net saving 6168200
B caught 900.0 false alarms 19980 net saving 6364800


## 9.5 Worked Design: RAG Chatbot for Company Documents

In [ ]:
chunks = 50_000 * 20; d = 768
print("chunks:", f"{chunks:,}")
print("vectors GB float32:", chunks * d * 4 / 1e9)
print("vectors GB int8   :", chunks * d / 1e9)

chunks: 1,000,000
vectors GB float32: 3.072
vectors GB int8   : 0.768


In [ ]:
q, tin, tout = 2000, 3000, 500
print("tokens/day:", q * (tin + tout), "in", q * tin, "out", q * tout)
print("tokens/month (M):", q * (tin + tout) * 30 / 1e6)
parts = {"embed query": 30, "vector search": 40, "rerank": 120, "LLM first token": 700}
first = sum(parts.values())
stream = tout / 50 * 1000
print("first token ms:", first, "| full answer s:", (first + stream) / 1000)
qps = q / (8 * 3600)
print("avg qps:", round(qps, 3), "peak x5:", round(qps * 5, 2), "streams open at peak (Little):", round(qps * 5 * (first + stream) / 1000, 1))

tokens/day: 7000000 in 6000000 out 1000000
tokens/month (M): 210.0
first token ms: 890 | full answer s: 10.89
avg qps: 0.069 peak x5: 0.35 streams open at peak (Little): 3.8


## Chapter recap

## Practice set

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/career/ch09.html